# Matching pennies: costly stochasticity (PC, small network)

Mice and a PC network play matching pennies against a competitive computer opponent, alternating with
coin-flip / random-reward phases (p(reward) = 0.5 for either choice).

**Model.** All 7 rules, small (3-neuron) network. One cell per rule, chosen from the settled-behaviour sweeps
(`data/sweeps/matchingpennies_steady_<RULE>_small.npz`; sweep code `ssb/scripts/run_mp_steady.py`;
20 seeds, base seed 825) by one rule: among cells whose behaviour has settled (repeat phases agree within 0.1 bits)
and that stay unbiased against the computer (median side bias < 0.05), the cell with the largest drop in 3-bit
entropy from competitive to random-reward phases. Each chosen cell is re-run here with 50 new seeds (base seed 235),
7 alternating phases of 20,000 steps (computer, then 3 x [random reward, computer]). One choice per timestep (no ITI),
so model "trials" = timesteps. Network state and the PRNG key carry across phases (fresh noise in every phase).
Phases 1-2 are the first exposure; phases 3-7 are treated as settled. Example traces, attractors and the mice
comparison use PC.

**Mice.** Aloor et al., public G-Node GIN release (`data/animals_mp/matchingpennies_datatable.parquet`, the full
47-mouse table; https://gin.g-node.org/DELab/matching-pennies-widefield-2026). Example traces: the two behavioural mice
JOA-M-0002 and JOA-M-0003, from the last MP session before the first coin-flip (CF) session on. Fig 3j: all seven mice
that alternated MP with CF (random-opponent, "Algorithm 0") sessions -- four behavioural (`CoinFlip`) and three
widefield (`CoinFlip_WF`) mice -- using each mouse's CF sessions and the MP sessions interleaved with and bracketing
them (sessions < 100 trials dropped). Missed trials dropped throughout.

**Metric.** 3-bit block entropy of the choice sequence (bits, max 3).

Panels (all written to `svgs/`):
1. `mp_mice_example.svg`, `mp_PC_example.svg`: example choice rasters and windowed entropy
2. `mp_scores.svg`: settled entropy per rule, and per mouse for the 7-mouse cohort
3. `mp_PC_attractors_late.svg`: W_diff vs X_diff, late random-reward vs late competitive
4. `mp_PC_attractors_sequence.svg`: all 7 phases (W_diff traces + attractors)
5. `mp_mice_vs_PC.svg`: entropy / side bias / win rate, mice per session vs model per phase

In [ ]:
import os
os.environ.setdefault('JAX_PLATFORMS', 'cpu')
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib as mpl, matplotlib.pyplot as plt, matplotlib.ticker as mticker
from matplotlib.collections import LineCollection
from matplotlib.lines import Line2D
mpl.rcParams.update({'font.family':'sans-serif','font.sans-serif':['Arial','Helvetica','DejaVu Sans'],'font.size':7,'svg.fonttype':'none',
                     'axes.linewidth':0.5,'xtick.major.width':0.5,'ytick.major.width':0.5,'xtick.major.size':2,'ytick.major.size':2,
                     'axes.spines.top':False,'axes.spines.right':False})

DATA  = Path('../../data/animals_mp/matchingpennies_datatable.parquet')
CACHE = Path('cache'); SVG = Path('svgs'); CACHE.mkdir(exist_ok=True); SVG.mkdir(exist_ok=True)
SIM   = lambda rule: CACHE/f'mp_seq_{rule}.npz'   # model simulations (written by the next section if missing)

C_COMP, C_RAND, TAG = '#3f5a8a', '#ea8d50', '#d76577'      # mice-vs-model panel
A_COMP, A_RAND = '#fbb040', '#508fcc'                         # attractor panels: computer, p(reward) = 0.5
MICE = ['JOA-M-0002', 'JOA-M-0003']

def nature_ax(ax, xlabel='', ylabel=''):
    for sp in ('top','right'): ax.spines[sp].set_visible(False)
    for sp in ('left','bottom'): ax.spines[sp].set_linewidth(0.25)
    ax.tick_params(labelsize=7, width=0.25)
    if xlabel: ax.set_xlabel(xlabel, fontsize=8)
    if ylabel: ax.set_ylabel(ylabel, fontsize=8)

def ent3(a):
    '''3-bit block entropy of one binary choice sequence (bits).'''
    b = a[:-2]*4 + a[1:-1]*2 + a[2:]; p = np.bincount(b, minlength=8)/len(b); p = p[p > 0]; return -(p*np.log2(p)).sum()
def ent3_seeds(A):
    '''A: (time, seeds) ints -> per-seed 3-bit entropy.'''
    return np.array([ent3(A[:, s]) for s in range(A.shape[1])])
def sliding(a, win):
    '''Trailing-window 3-bit entropy; output[i] uses a[i : i+win].'''
    return np.array([ent3(a[i-win:i]) for i in range(win, len(a)+1)])

## 1. Simulations (skipped for rules whose cache exists)

Per-step loop identical to `ssb.scripts.run.run_env` (env -> expand -> update_acts -> noise -> update_weights ->
update_states), except that the PRNG key is carried across phases. Logged per step and seed:
x_in, W_diff = w_a - w_b, X_diff = x_a - x_b, action, reward. ~25 s per rule on CPU.

In [ ]:
# chosen cell per rule (eta, tau, alpha, beta); tau is unused by PC, HR and OJA
CELLS = {'BCM': dict(eta=1e-4, tau=0.001,   alpha=0.0372759, beta=1.0),
         'OJA': dict(eta=1e-2, tau=0.01,    alpha=0.0013895, beta=0.0013895),
         'LC':  dict(eta=1e-6, tau=0.0215443, alpha=0.439397, beta=0.0848343),
         'PC':  dict(eta=1e-3, tau=0.01,    alpha=0.0163789, beta=0.439397),
         'COV': dict(eta=1e-6, tau=0.001,   alpha=0.0031623, beta=0.0372759),
         'HZ':  dict(eta=0.1,  tau=0.1,     alpha=1.0,       beta=1.0),
         'HR':  dict(eta=1e-6, tau=0.01,    alpha=0.0848343, beta=1.0)}   # figure order; HZ = LPL, HR = RPE
N = 50; T = 20000; BASE_SEED = 235
SEQ = ['computer', 'random', 'computer', 'random', 'computer', 'random', 'computer']

def simulate(rule, P):
    import jax, jax.numpy as jnp
    from ssb.utils import batch_inits
    from ssb.models import ssb, rules
    from ssb.envs import matchingpennies as mp, bandits
    hps = batch_inits.make_default_hps(seed=BASE_SEED)          # default sizes = small network
    for k, v in [('alpha','alpha'), ('beta','beta'), ('eta_a','eta'), ('tau','tau')]: hps[k] = jnp.array([P[v]])
    hb = batch_inits.batchify_hps(hps, batch_size=N)
    uw, us = rules.rule2fns(rule); UA = jax.vmap(ssb.update_acts); UW = jax.vmap(uw); US = jax.vmap(us)
    NO = jax.vmap(ssb.act_noise); EX = jax.vmap(ssb.expand_stimulus)
    acts, states, weights, tern, key = batch_inits.init_params_batch(hb); out = []
    ENV = {'computer': (mp.play_mp, jnp.array([0.05])), 'random': (bandits.bandit_unbait, jnp.array([0.5, 0.5]))}
    for ph in SEQ:
        fn, par = ENV[ph]; envb = jax.vmap(fn); parb = jnp.tile(par[None], (N, 1))
        es = jax.tree_util.tree_map(lambda x: jnp.tile(x[None], (N,) + (1,)*x.ndim), mp.init_mp_state(100))   # opponent history resets
        def step(c, _):
            es, acts, states, weights, key = c
            r, a, es, key = envb(es, acts[:, -2:], parb, key)
            stim = EX(r, acts); acts = UA(stim, acts, states, weights, tern, hb); acts, key = NO(acts, key, hb)
            weights = UW(stim, acts, states, weights, tern, hb); states = US(acts, states, hb)
            return (es, acts, states, weights, key), jnp.stack([acts[:, 0], weights[:, 0, 2]-weights[:, 0, 1], acts[:, 2]-acts[:, 1],
                                                              a.astype(jnp.float32), r[:, 0].astype(jnp.float32)], 1)
        (es, acts, states, weights, key), o = jax.jit(lambda c: jax.lax.scan(step, c, None, length=T))((es, acts, states, weights, key))
        out.append(np.asarray(o))
    return np.stack(out).astype(np.float32)

DATA_ALL = {}
for rule, P in CELLS.items():
    if not SIM(rule).exists():
        np.savez_compressed(SIM(rule), seq=np.array(SEQ), data=simulate(rule, P), **{k: np.float64(v) for k, v in P.items()})
    DATA_ALL[rule] = np.load(SIM(rule))['data']
    A = [DATA_ALL[rule][j][T//2:, :, 3].astype(int) for j in range(len(SEQ))]
    print(f'{rule:4s} entropy by phase', np.round([np.median(ent3_seeds(a)) for a in A], 2),
          ' side bias', np.round([np.median(np.abs(a.mean(0)-.5)) for a in A], 2))

D = DATA_ALL['PC']                                 # PC: example traces, attractors, mice comparison
acts = lambda j: D[j][:, :, 3].astype(int)       # (time, seeds) choices of phase j (0-based)
COMP, RAND = [2, 4, 6], [3, 5]                     # settled phases: competitive 3,5,7; random 4,6
for j, ph in enumerate(SEQ):
    A = acts(j)[T//2:]; e = ent3_seeds(A)
    print(f'PC phase {j+1} {ph:8s}: entropy median {np.median(e):.2f}, side bias {np.median(np.abs(A.mean(0)-.5)):.2f}, '
          f'P(switch) {np.median((A[1:] != A[:-1]).mean(0)):.3f}, win {np.median(D[j][T//2:, :, 4].mean(0)):.2f}')

LC lowers its entropy under random reward without an overall side bias: it makes long runs on one side and then
switches (P(switch) ~0.03, mean run ~30 trials), so it is locally biased (side bias ~0.24 in 300-trial blocks) but
balanced over the whole phase. Against the computer it switches on ~37% of trials, close to the mice (~39%).

In [ ]:
for rule in ['LC']:
    for j, name in [(6, 'competitive (phase 7)'), (5, 'random reward (phase 6)')]:
        A = DATA_ALL[rule][j][T//2:, :, 3].astype(int)
        runs = np.concatenate([np.diff(np.flatnonzero(np.diff(np.r_[-1, A[:, s], -1]) != 0)) for s in range(N)])
        blk = np.abs(A[:A.shape[0]//300*300].reshape(-1, 300, N).mean(1)-.5)
        print(f'{rule} {name}: P(switch) {np.median((A[1:] != A[:-1]).mean(0)):.3f}, mean run {runs.mean():.1f}, '
              f'side bias {np.median(np.abs(A.mean(0)-.5)):.2f} (whole phase) / {np.median(blk):.2f} (300-trial blocks)')

## 2. Mouse data

In [ ]:
DF = pd.read_parquet(DATA); DF = DF[DF.choice != 'M']                    # public table, all protocols
df = DF[DF.protocol.isin(['MatchingPennies', 'CoinFlip'])]               # example mice: behavioural MP + CF sessions
MS = []                                              # one row per session in the alternating period
for m in MICE:
    g = df[df.subjid == m]
    ss = g.groupby('sessid').agg(date=('date','first'), protocol=('protocol','first')).sort_values('date').reset_index()
    ss = ss.iloc[ss.index[ss.protocol == 'CoinFlip'][0]-1:]
    for _, r in ss.iterrows():
        x = g[g.sessid == r.sessid].sort_values('trialnum'); a = (x.choice == 'R').astype(int).values
        MS.append(dict(mouse=m, sessid=r.sessid, cf=r.protocol == 'CoinFlip', a=a, H=ent3(a), bias=abs(a.mean()-.5),
                       switch=(a[1:] != a[:-1]).mean(), win=x.reward.mean()))
MS = pd.DataFrame(MS)
print(MS.groupby(['mouse', 'cf'])[['H', 'bias', 'switch', 'win']].median().round(2))

# Fig 3j cohort: every mouse that alternated MP sessions with coin-flip (random opponent, "Algorithm 0") sessions --
# 4 behavioural (CoinFlip) and 3 widefield (CoinFlip_WF) mice. Per mouse: its CF sessions and the MP sessions interleaved
# with and immediately bracketing them; sessions < 100 trials (restart fragments) dropped. The 'MatchingPennies_CoinFlip'
# protocol (earlier sessions of the behavioural mice) is not used.
from scipy.stats import wilcoxon
MIN_TRIALS = 100; COHORT = []
for cfp, mpp in [('CoinFlip', 'MatchingPennies'), ('CoinFlip_WF', 'MatchingPennies_WF')]:
    for m in sorted(DF.loc[DF.protocol == cfp, 'subjid'].unique()):
        g = DF[DF.subjid == m]
        ss = g.groupby('sessid').agg(date=('date','first'), protocol=('protocol','first'), n=('choice','size')).reset_index()
        ss = ss[ss.n >= MIN_TRIALS]
        cfd, mpd = ss.date[ss.protocol == cfp], ss.date[ss.protocol == mpp]
        lo, hi = mpd[mpd < cfd.min()].max(), mpd[mpd > cfd.max()].min()
        hi = hi if pd.notna(hi) else cfd.max()
        for _, r in ss[(ss.date >= lo) & (ss.date <= hi) & ss.protocol.isin([cfp, mpp])].iterrows():
            a = (g[g.sessid == r.sessid].sort_values('trialnum').choice == 'R').astype(int).values
            COHORT.append(dict(mouse=m, cf=r.protocol == cfp, H=ent3(a)))
MOUSE_H = pd.DataFrame(COHORT).groupby(['mouse', 'cf']).H.mean().unstack()   # columns: False = vs computer, True = coin flip
print(MOUSE_H.round(2))
print(f"n = {len(MOUSE_H)} mice: vs computer {MOUSE_H[False].mean():.2f} +/- {MOUSE_H[False].sem():.2f}, "
      f"coin flip {MOUSE_H[True].mean():.2f} +/- {MOUSE_H[True].sem():.2f} (mean +/- s.e.m.); one-sided paired Wilcoxon "
      f"p = {wilcoxon(MOUSE_H[False], MOUSE_H[True], alternative='greater').pvalue:.3g}")


## 3. Example traces

Model shown over 10x more trials than mice (3,000 vs 300) to allow for its slower timescale; the entropy window is
scaled by the same factor (300 vs 30 trials). Mouse examples, among sessions with enough trials (>= 329): vs
computer, the session closest to the median mean windowed entropy; coin flip, a clearly biased session that still
switches between stretches (JOA-M-0003 session 463, third-lowest mean windowed entropy of 8; the two lower sessions
are almost entirely one-sided). Model example: the seed closest to the median settled
entropy in both conditions. Competitive: last 3,000 trials of phase 7. Random reward: among that seed's 3,000-trial
segments (2nd half of phases 4 and 6, step 500), those whose side preference (300-trial blocks) changes at least
once (58% of all seeds' segments do), and of those the one whose mean windowed entropy is closest to the median
over all seeds' segments.

In [ ]:
WIN_M, SHOW_M = 30, 300
SHOW_PC = 3000; WIN_PC = WIN_M * SHOW_PC // SHOW_M

def example(seqs, fname, show, win, tags=('Stochastic', 'Biased')):
    fig, axes = plt.subplots(4, 1, figsize=(1.75, 2.9), gridspec_kw=dict(height_ratios=[0.35, 1, 0.35, 1], hspace=0.45))
    for k, (a, lab, tag) in enumerate(zip(seqs, ['vs computer', 'p(reward) = 0.5'], tags)):
        a = np.asarray(a)[-(show+win-1):]; h = sliding(a, win); ch = a[win-1:]; t = np.arange(show)
        ax = axes[2*k]; lw = 0.4 if show <= 500 else 0.15
        ax.vlines(t[ch == 1], 0.55, 1, color='k', lw=lw); ax.vlines(t[ch == 0], 0, 0.45, color='k', lw=lw)
        ax.set_ylim(0, 1); ax.set_xlim(0, show); ax.set_yticks([0.25, 0.75]); ax.set_yticklabels(['T', 'H'])
        ax.tick_params(length=0, labelbottom=False); ax.spines['bottom'].set_visible(False); ax.spines['left'].set_visible(False)
        ax.set_title(lab, fontsize=7, pad=2)
        ax = axes[2*k+1]; ax.plot(t, h, color='k', lw=0.6); ax.set_ylim(0, 3.05); ax.set_yticks([0, 3]); ax.set_xlim(0, show)
        if tag: ax.text(show*1.02, 0.4 if k else 2.4, tag, color=TAG, fontsize=6.5, va='center')
        if k == 0: ax.tick_params(labelbottom=False)
        else: ax.set_xticks([0, show]); ax.set_xticklabels([0, f'{show:,}']); ax.set_xlabel('Trials', labelpad=1)
    fig.text(-0.02, 0.5, '3-bit entropy', rotation=90, va='center', fontsize=7)
    fig.savefig(SVG/f'{fname}.svg', bbox_inches='tight'); plt.show()

# mice
pick = {}
for cf in (False, True):
    L = MS[(MS.cf == cf) & (MS.a.apply(len) >= SHOW_M+WIN_M-1)].copy()
    L['wH'] = [sliding(a[-(SHOW_M+WIN_M-1):], WIN_M).mean() for a in L.a]
    pick[cf] = L.iloc[int(np.argmin(np.abs(L.wH-L.wH.median())))]
    print(('CF' if cf else 'MP'), f'{len(L)} eligible sessions, median windowed H {L.wH.median():.2f}; '
          f'median session {pick[cf].mouse} {pick[cf].sessid} ({pick[cf].wH:.2f}), side bias {pick[cf].bias:.2f}')
CF_EXAMPLE = 463                                     # chosen coin-flip example (see markdown above)
L = MS[(MS.cf) & (MS.a.apply(len) >= SHOW_M+WIN_M-1)].copy(); L['wH'] = [sliding(a[-(SHOW_M+WIN_M-1):], WIN_M).mean() for a in L.a]
pick[True] = L[L.sessid == CF_EXAMPLE].iloc[0]
print(f'CF example used: {pick[True].mouse} session {pick[True].sessid}, windowed H {pick[True].wH:.2f} '
      f'(rank {int((L.wH < pick[True].wH).sum())+1} of {len(L)} from most biased), side bias {pick[True].bias:.2f}')
example([pick[False].a, pick[True].a], 'mp_mice_example', SHOW_M, WIN_M)

# model
hc, hr = ent3_seeds(acts(6)[T//2:]), ent3_seeds(acts(5)[T//2:])
seed = int(np.argmin(np.abs(hc-np.median(hc))/hc.std() + np.abs(hr-np.median(hr))/hr.std()))
def segments(s, phases):
    out = []
    for j in phases:
        a = acts(j)[T//2:, s]
        for st in range(WIN_PC-1, len(a)-SHOW_PC+1, 500):
            seg = a[st-WIN_PC+1:st+SHOW_PC]; blk = a[st:st+SHOW_PC].reshape(-1, 300).mean(1)
            out.append((j, st, int(((blk[1:] > .5) != (blk[:-1] > .5)).sum()), sliding(seg, WIN_PC)[::10].mean(), seg))
    return out
allseg = [x for s in range(N) for x in segments(s, RAND)]; med_r = np.median([x[3] for x in allseg])
ex_r = min([x for x in segments(seed, RAND) if x[2] >= 1], key=lambda x: abs(x[3]-med_r))
print(f'model seed {seed}; random-reward segments with a side switch: {np.mean([x[2] >= 1 for x in allseg]):.2f}; '
      f'example phase {ex_r[0]+1}, trials {ex_r[1]}-{ex_r[1]+SHOW_PC} of the 2nd half, windowed H {ex_r[3]:.2f} (median {med_r:.2f})')
example([acts(6)[:, seed], ex_r[4]], 'mp_PC_example', SHOW_PC, WIN_PC)

## 4. Settled entropy per rule

Each seed's entropy over the 2nd half of each settled phase, averaged across phases of a type; mean +/- s.e.m. over
50 seeds.
Mice (left column): each mouse's mean over sessions of each type, mean +/- s.e.m. over the 7 mice; thin lines join
each mouse's two values.


In [ ]:
SCORES = {}
for rule, DR in DATA_ALL.items():
    Hc = np.mean([ent3_seeds(DR[j][T//2:, :, 3].astype(int)) for j in COMP], 0); Hr = np.mean([ent3_seeds(DR[j][T//2:, :, 3].astype(int)) for j in RAND], 0)
    SCORES[rule] = (Hc, Hr)
    print(f'{rule:4s}: competitive {Hc.mean():.2f} +/- {Hc.std(ddof=1)/np.sqrt(N):.2f}, random {Hr.mean():.2f} +/- {Hr.std(ddof=1)/np.sqrt(N):.2f}')
ORDER = ['BCM', 'OJA', 'LC', 'PC', 'COV', 'HZ', 'HR']; NAME = {'HR': 'RPE', 'HZ': 'LPL'}
COL = {'HR':'#74ad83','PC':'#7991b5','BCM':'#2f3f97','OJA':'#e8894a','LC':'#967fb7','COV':'#c0466a','HZ':'#e5a93b'}
X0 = 1.3                                                    # rules start here; mice at x = 0
fig, ax = plt.subplots(figsize=(2.2, 1.6))
hm = (MOUSE_H[False].values, MOUSE_H[True].values)
for a, b in zip(*hm): ax.plot([-0.18, 0.18], [a, b], color='#d0d0d0', lw=0.4, zorder=0)
ax.scatter(np.full(len(hm[0]), -0.18), hm[0], s=3, c='k', lw=0, zorder=1)
ax.scatter(np.full(len(hm[1]), 0.18), hm[1], s=3, facecolors='white', edgecolors='k', linewidths=0.4, zorder=1)
for v, filled in zip(hm, (True, False)):
    ax.errorbar(0, v.mean(), yerr=v.std(ddof=1)/np.sqrt(len(v)), fmt='o', ms=4, mfc='k' if filled else 'white', mec='k',
                ecolor='k', elinewidth=0.8, capsize=2.5, mew=0.8, zorder=2)
ax.axvline(0.65, color='#bbbbbb', lw=0.5, ls=':')
for i, r in enumerate(ORDER):
    if r not in SCORES: continue
    for v, filled in zip(SCORES[r], (True, False)):
        ax.errorbar(X0 + i, v.mean(), yerr=v.std(ddof=1)/np.sqrt(len(v)), fmt='o', ms=4, mfc=COL[r] if filled else 'white', mec=COL[r],
                    ecolor=COL[r], elinewidth=0.8, capsize=2.5, mew=0.8)
ax.set_xlim(-0.5, X0+len(ORDER)-0.4); ax.set_xticks([0] + [X0 + i for i in range(len(ORDER))])
ax.set_xticklabels(['Mice'] + [NAME.get(r, r) for r in ORDER], rotation=45, ha='right')
ax.set_ylim(-0.1, 3.1); ax.set_yticks([0, 1, 2, 3]); ax.set_ylabel('3-bit entropy'); ax.spines['bottom'].set_visible(False); ax.tick_params(axis='x', length=0)
ax.axhline(0, color='k', lw=0.5)
h1 = ax.scatter([], [], s=12, c='k', label='vs computer'); h2 = ax.scatter([], [], s=12, facecolors='white', edgecolors='k', linewidths=0.8, label='p(rew)=0.5')
ax.legend(handles=[h1, h2], frameon=False, ncol=2, loc='lower center', bbox_to_anchor=(0.5, 1.0), handletextpad=0.1, columnspacing=0.8, fontsize=6.5)
fig.savefig(SVG/'mp_scores.svg', bbox_inches='tight'); plt.show()

## 5. Attractors, late random reward vs late competitive

W_diff vs X_diff over the last 5,000 steps of phase 6 (random reward) and phase 7 (competitive), all 50 seeds;
grey = the other condition.

In [ ]:
CLOUD, NPTS = 5000, 3000; WL, XL = 0.3, 0.15; rng = np.random.default_rng(0)   # axes cover ~99% of random-reward points
cloud = lambda j: (D[j][-CLOUD:, :, 1].ravel(), D[j][-CLOUD:, :, 2].ravel())
fig, axes = plt.subplots(1, 2, figsize=(3.0, 1.6), gridspec_kw=dict(wspace=0.15))
for ci, (j, other, c, title) in enumerate([(5, 6, A_RAND, 'p(reward) = 0.5'), (6, 5, A_COMP, 'vs computer')]):
    ax = axes[ci]; w, x = cloud(j); wo, xo = cloud(other)
    ax.plot([-XL, XL], [-XL, XL], color='#ececec', lw=0.7, zorder=0)
    i = rng.choice(wo.size, NPTS, replace=False); ax.scatter(wo[i], xo[i], s=2.5, c='#d6d6d6', lw=0, alpha=0.6, zorder=1, rasterized=True)
    i = rng.choice(w.size, NPTS, replace=False); ax.scatter(w[i], x[i], s=3.5, c=c, lw=0, alpha=0.5, zorder=2, rasterized=True)
    off = (np.abs(D[j][-CLOUD:, :, 1]).max(0) > WL).sum()
    ax.text(0.97, 0.05, f'|W$_{{diff}}$| {np.median(np.abs(w)):.2f}' + (f'\n{off}/{N} off-axis' if off else ''), transform=ax.transAxes,
            ha='right', va='bottom', fontsize=6, color=c)
    ax.set_xlim(-WL, WL); ax.set_ylim(-XL, XL); ax.set_xticks([-0.3, 0, 0.3]); ax.set_yticks([-0.15, 0, 0.15])
    ax.set_title(title, fontsize=7, color=c, pad=3); ax.set_xlabel('W$_{diff}$', labelpad=1)
    if ci == 0: ax.set_ylabel('X$_{diff}$', labelpad=1)
    else: ax.tick_params(labelleft=False)
fig.savefig(SVG/'mp_PC_attractors_late.svg', bbox_inches='tight', dpi=300); plt.show()

## 6. Full sequence: W_diff traces and attractors for all 7 phases

Top: W_diff per seed (thin) and median (black), symlog axis. Bottom: last 5,000 steps of each phase; grey = the
previous phase. Titles give the median entropy over the 2nd half of each phase.

In [ ]:
NAMEP = {'computer': 'Competitive', 'random': 'Random reward'}; STRIDE = 20; rng = np.random.default_rng(0); n = len(SEQ)
fig, axes = plt.subplots(2, n, figsize=(1.45*n+0.6, 3.3), gridspec_kw=dict(height_ratios=[0.8, 1], hspace=0.55, wspace=0.18))
prev = None
for j, ph in enumerate(SEQ):
    o = D[j]; c = A_COMP if ph == 'computer' else A_RAND; H = np.median(ent3_seeds(acts(j)[T//2:]))
    ax = axes[0, j]; W = o[::STRIDE, :, 1]; t = np.arange(0, T, STRIDE)/1e3
    ax.add_collection(LineCollection([np.column_stack([t, W[:, s]]) for s in range(W.shape[1])], colors=[c], linewidths=0.25, alpha=0.35))
    ax.plot(t, np.median(W, 1), color='#222', lw=0.9); ax.axhline(0, color='#999', lw=0.4, ls='--')
    ax.set_yscale('symlog', linthresh=1e-5, linscale=0.6); ax.set_ylim(-1, 1); ax.set_xlim(0, T/1e3)
    ax.set_yticks([-1, -1e-2, -1e-4, 0, 1e-4, 1e-2, 1]); ax.yaxis.set_minor_locator(mticker.NullLocator())
    nature_ax(ax, 'step (×10$^3$)' if j == n//2 else '', 'W$_{diff}$' if j == 0 else '')
    if j: ax.tick_params(labelleft=False)
    ax.set_title(f'{j+1}. {NAMEP[ph]}\n{H:.2f} bits', fontsize=8, color=c)
    ax = axes[1, j]; s = o[-CLOUD:]; w = s[..., 1].ravel(); x = s[..., 2].ravel()
    ax.plot([-XL, XL], [-XL, XL], color='#ececec', lw=0.7, zorder=0)
    if prev is not None:
        i = rng.choice(prev[0].size, 3000, replace=False); ax.scatter(prev[0][i], prev[1][i], s=2.5, c='#d6d6d6', lw=0, alpha=0.6, zorder=1, rasterized=True)
    i = rng.choice(w.size, 3000, replace=False); point = np.median(np.abs(w)) < 1e-4
    ax.scatter(w[i], x[i], s=6 if point else 3.5, c=c, lw=0, alpha=1 if point else 0.5, zorder=7 if point else 2, rasterized=True)
    off = (np.abs(s[..., 1]).max(0) > WL).sum()
    ax.text(0.97, 0.05, f'|W$_{{diff}}$| {np.median(np.abs(w)):.2f}' + (f'\n{off}/{N} off-axis' if off else ''), transform=ax.transAxes,
            ha='right', va='bottom', fontsize=6, color=c)
    ax.set_xlim(-WL, WL); ax.set_ylim(-XL, XL)
    nature_ax(ax, 'W$_{diff}$' if j == n//2 else '', 'X$_{diff}$' if j == 0 else '')
    ax.set_xticks([-0.3, 0, 0.3]); ax.set_yticks([-0.15, 0, 0.15])
    if j > 0: ax.tick_params(labelleft=False)
    prev = (w, x)
axes[1, -1].legend(handles=[Line2D([0], [0], marker='o', color='none', markerfacecolor='#d6d6d6', markeredgecolor='none', markersize=4, label='previous phase')],
                   fontsize=6, frameon=False, loc='upper left', bbox_to_anchor=(1.02, 1.0), handletextpad=0.3)
fig.savefig(SVG/'mp_PC_attractors_sequence.svg', bbox_inches='tight', dpi=300); plt.show()

## 7. Mice vs model: entropy, side bias, win rate

Left: one point per session (two mice). Right: model, median and IQR over 50 seeds of each phase's 2nd half;
phase 1 (first exposure) drawn open.

In [ ]:
mod = []
for j, ph in enumerate(SEQ):
    s = D[j][T//2:]; A = s[..., 3].astype(int)
    mod.append(dict(cf=ph == 'random', H=ent3_seeds(A), bias=np.abs(A.mean(0)-.5), win=s[..., 4].mean(0)))
ROWS = [('H', '3-bit entropy (bits)', (0, 3.1)), ('bias', 'side bias', (0, 0.52)), ('win', 'win rate', (0, 0.62))]
fig, axes = plt.subplots(3, 2, figsize=(4.6, 4.0), gridspec_kw=dict(width_ratios=[1.6, 1], hspace=0.35, wspace=0.12))
for r, (k, lab, lim) in enumerate(ROWS):
    ax = axes[r, 0]
    for m, mk in zip(MICE, ['o', 's']):
        d = MS[MS.mouse == m].reset_index(drop=True); x = np.arange(len(d))
        ax.plot(x, d[k], color='#bbbbbb', lw=0.6, zorder=1)
        ax.scatter(x, d[k], c=[C_RAND if c else C_COMP for c in d.cf], marker=mk, s=12, lw=0, zorder=2)
    ax.set_ylim(*lim); ax.set_xlim(-0.5, (MS.mouse == MICE[0]).sum()-0.5)
    nature_ax(ax, 'session' if r == 2 else '', lab)
    if r < 2: ax.tick_params(labelbottom=False)
    ax = axes[r, 1]
    for j, p in enumerate(mod):
        q = np.percentile(p[k], [25, 50, 75]); c = C_RAND if p['cf'] else C_COMP
        ax.plot([j, j], [q[0], q[2]], color=c, lw=0.8, zorder=1)
        ax.scatter([j], [q[1]], s=14, facecolors='white' if j == 0 else c, edgecolors=c, lw=0.8, zorder=2)
    ax.plot(range(len(mod)), [np.median(p[k]) for p in mod], color='#bbbbbb', lw=0.6, zorder=0)
    ax.set_ylim(*lim); ax.set_xlim(-0.5, len(mod)-0.5); ax.set_xticks(range(len(mod))); ax.set_xticklabels(range(1, len(mod)+1))
    nature_ax(ax, 'phase (20k steps)' if r == 2 else ''); ax.tick_params(labelleft=False)
    if r < 2: ax.tick_params(labelbottom=False)
    for a_ in axes[r]:
        if k == 'H': a_.axhline(3, color='#999', lw=0.5, ls='--', zorder=0)
        if k == 'win': a_.axhline(0.5, color='#999', lw=0.5, ls='--', zorder=0)
axes[0, 0].set_title('Mice (n = 2)', fontsize=8); axes[0, 1].set_title(f'PC model ({N} seeds)', fontsize=8)
axes[0, 0].text(0.0, 3.05, 'maximally random', fontsize=6, color='#999', va='bottom')
h = [Line2D([0], [0], marker='o', color='none', markerfacecolor='#888', markeredgecolor='none', markersize=4, label=f'mouse {MICE[0]}'),
     Line2D([0], [0], marker='s', color='none', markerfacecolor='#888', markeredgecolor='none', markersize=4, label=f'mouse {MICE[1]}'),
     Line2D([0], [0], marker='o', color='none', markerfacecolor=C_COMP, markeredgecolor='none', markersize=4, label='vs computer'),
     Line2D([0], [0], marker='o', color='none', markerfacecolor=C_RAND, markeredgecolor='none', markersize=4, label='coin flip / random reward')]
axes[0, 1].legend(handles=h, fontsize=6, frameon=False, loc='upper left', bbox_to_anchor=(1.02, 1.0))
fig.savefig(SVG/'mp_mice_vs_PC.svg', bbox_inches='tight'); plt.show()
print('model', {('rand' if p['cf'] else 'comp')+str(j+1): tuple(round(float(np.median(p[k_])), 2) for k_ in ('H', 'bias', 'win')) for j, p in enumerate(mod)})